# K-Means Clustering — Student Data Application

## What We're Doing
Apply the custom **KMeans class** (built from scratch) to real student data.
The dataset contains student records — we'll cluster them into 4 groups to find natural groupings.

## Goal
Discover natural student segments based on their features, without using any labels.
This is **unsupervised learning** — we let the algorithm find structure in the data.


## Step 1: Imports

In [ ]:
from sklearn.datasets import make_blobs
import matplotlib.pyplot as plt
import pandas as pd
import random
import numpy as np

## Step 2: KMeans Class (from scratch)
The complete KMeans implementation — same as `kmeans.py` but embedded here so the notebook is self-contained.


In [ ]:
class KMeans:
    def __init__(self, n_clusters=2, max_iter=100):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.centroids = None

    def fit_predict(self, X):
        random_index = random.sample(range(0, X.shape[0]), self.n_clusters)
        self.centroids = X[random_index]

        for i in range(self.max_iter):
            cluster_group = self.assign_clusters(X)
            old_centroids = self.centroids
            self.centroids = self.move_centroids(X, cluster_group)
            if (old_centroids == self.centroids).all():
                break

        return cluster_group

    def assign_clusters(self, X):
        cluster_group = []
        distances = []
        for row in X:
            for centroid in self.centroids:
                distances.append(np.sqrt(np.dot(row - centroid, row - centroid)))
            min_distance = min(distances)
            index_pos = distances.index(min_distance)
            cluster_group.append(index_pos)
            distances.clear()
        return np.array(cluster_group)

    def move_centroids(self, X, cluster_group):
        new_centroids = []
        cluster_type = np.unique(cluster_group)
        for type in cluster_type:
            new_centroids.append(X[cluster_group == type].mean(axis=0))
        return np.array(new_centroids)

## Step 3: Load the Student Dataset
Read `student_clustering.csv` — a dataset of student records with 2 numeric features.


In [ ]:
df = pd.read_csv(r'd:\PENDRIVE 32 GB\Game\AI_PROJECTS\Machine Learning\100-days-of-machine-learning-main\kmeans\student_clustering.csv')
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
df.head()

## Step 4: Explore the Data
Check basic statistics and visualise the raw data before clustering.


In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(df.iloc[:, 0], df.iloc[:, 1], alpha=0.6, edgecolors='k', linewidths=0.3)
plt.xlabel(df.columns[0])
plt.ylabel(df.columns[1])
plt.title('Student Data — Before Clustering')
plt.tight_layout()
plt.show()

## Step 5: Prepare Feature Matrix
Extract all columns as a NumPy array — KMeans works on numeric arrays.


In [ ]:
X = df.iloc[:, :].values
print(f"Feature matrix shape: {X.shape}")

## Step 6: Fit KMeans — 4 Clusters
We choose **K=4** clusters, `max_iter=500` to ensure convergence.

**Why K=4?**
Domain knowledge about student groupings (e.g. high/low performance × high/low engagement).
To find the optimal K objectively, use the **Elbow Method** (see Step 8).


In [ ]:
km = KMeans(n_clusters=4, max_iter=500)
y_means = km.fit_predict(X)

print(f"Cluster labels: {y_means}")
print(f"Unique clusters: {set(y_means)}")
print(f"\nFinal centroids:\n{km.centroids}")

## Step 7: Visualise the Clusters
Each colour = one cluster. The cluster boundaries are determined by the learned centroids.


In [ ]:
colors = ['red', 'blue', 'green', 'orange']
labels = ['Cluster 0', 'Cluster 1', 'Cluster 2', 'Cluster 3']

plt.figure(figsize=(8, 6))
for i in range(4):
    plt.scatter(X[y_means == i, 0], X[y_means == i, 1],
                color=colors[i], label=labels[i], alpha=0.7, edgecolors='k', linewidths=0.3)

# Plot centroids
plt.scatter(km.centroids[:, 0], km.centroids[:, 1],
            marker='X', s=250, c='black', zorder=5, label='Centroids')

plt.xlabel(df.columns[0] if len(df.columns) > 0 else 'Feature 1')
plt.ylabel(df.columns[1] if len(df.columns) > 1 else 'Feature 2')
plt.title('KMeans Clustering — Student Data (K=4)')
plt.legend()
plt.tight_layout()
plt.show()

## Step 8: Elbow Method — Finding the Optimal K
The **Elbow Method** plots WCSS (Within-Cluster Sum of Squares) vs K.

```
WCSS = Σ Σ ||xᵢ - centroid_k||²
       k  xᵢ∈k
```

- As K increases, WCSS always decreases (more clusters = tighter fit)
- The **elbow point** is where the rate of decrease slows sharply
- That K is usually the best trade-off between simplicity and fit

We use **sklearn's KMeans** here (faster) just to compute WCSS across K values.


In [ ]:
from sklearn.cluster import KMeans as SKLearnKMeans

wcss = []
K_range = range(1, 11)
for k in K_range:
    skm = SKLearnKMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    skm.fit(X)
    wcss.append(skm.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, wcss, marker='o', linewidth=2)
plt.axvline(x=4, color='red', linestyle='--', alpha=0.7, label='K=4 (chosen)')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Inertia)')
plt.title('Elbow Method — Optimal K')
plt.legend()
plt.tight_layout()
plt.show()

## Step 9: Cluster Sizes
How many students fell into each cluster?


In [ ]:
import numpy as np

unique, counts = np.unique(y_means, return_counts=True)
for cluster, count in zip(unique, counts):
    print(f"Cluster {cluster}: {count} students ({count/len(y_means)*100:.1f}%)")

## Summary

```
K-Means on Student Data:

  Dataset:    student_clustering.csv (2D features)
  K:          4 clusters
  max_iter:   500
  Init:       Random (our scratch implementation)

  Results:
    - 4 distinct student segments identified
    - Centroids show the "average" student in each group
    - Elbow method confirms K=4 is reasonable

  Next steps:
    - Try KMeans++ initialisation (sklearn default) for better stability
    - Scale features with StandardScaler if units differ
    - Use Silhouette Score for a more rigorous K selection
    - Name the clusters based on domain knowledge
```
